# C4 · LLM endpoint and grounded prompt (live session)

Checks the configured OpenAI-compatible endpoint, then prototypes the **grounded prompt**: the LLM only phrases an
answer from tool output (evidence lines with ids) and must say it doesn't know when the evidence doesn't cover the
question (PLAN drift item 14). The agreed prompt moves to `anchorai/assistant/prompt.py`.

Before starting: put your key in `.env` as `LLM_API_KEY`.

## 1. Endpoint and models

In [1]:
import re
import time
import json

import pandas as pd

from anchorai.config import get_settings

# Settings are cached; clear after editing .env so changes are picked up.
get_settings.cache_clear()
s = get_settings()
# Print only whether things are set: the endpoint and key are private and this notebook is committed.
print("endpoint:", "set" if s.llm_base_url else "(not set)")
print("model:   ", s.llm_model or "(not set)")
print("key:     ", "set" if s.llm_api_key and "PASTE" not in s.llm_api_key else "(not set)")

endpoint: set
model:    claude-haiku-5-5
key:      set


In [2]:
from anchorai.assistant.llm import get_client

models = sorted(m.id for m in get_client().models.list())
print(len(models), "models")
models

29 models


['claude-3-5-haiku-20241022',
 'claude-3-7-sonnet-20250219',
 'claude-fable-5',
 'claude-fable-5-1',
 'claude-haiku-4-5-20251001',
 'claude-haiku-5-5',
 'claude-opus-4-1-20250805',
 'claude-opus-4-20250514',
 'claude-opus-4-5-20251101',
 'claude-opus-4-6',
 'claude-opus-4-7',
 'claude-opus-4-8',
 'claude-opus-5',
 'claude-opus-5-5',
 'claude-sonnet-4-20250514',
 'claude-sonnet-4-5-20250929',
 'claude-sonnet-4-6',
 'claude-sonnet-5',
 'claude-sonnet-5-5',
 'codex-auto-review',
 'gpt-5.5',
 'gpt-5.6-luna',
 'gpt-5.6-terra',
 'gpt-6-luna',
 'gpt-image-1.5',
 'gpt-image-2',
 'gpt-image-2.5',
 'gpt-image-2.5-flare',
 'gpt-image-2.5-sunburst']

Pick a model from the list, set `LLM_MODEL=...` in `.env`, then **re-run the first code cell**. A fast,
instruction-following chat model is enough: it only rewrites evidence into a sentence or two.

In [3]:
from anchorai.assistant.llm import chat

times = []
for _ in range(3):
    start = time.perf_counter()
    reply = chat([{"role": "user", "content": "Reply with exactly: OK"}])
    times.append(time.perf_counter() - start)
print(repr(reply), "| latency (s):", [round(t, 2) for t in times])

'OK' | latency (s): [0.95, 0.71, 0.73]


## 2. Real evidence for one member

The evidence the assistant will see comes from the same services the API uses: churn score and drivers (Stream A),
rule trace (C1/C2) and KG signals (C3). Each line gets an id so the answer can cite it.

In [4]:
from anchorai.discovery.scores import load_features, load_scores
from anchorai.features import LABELS
from anchorai.knowledge.signals import graph_signals, load_kg_facts
from anchorai.reasoning.engine import load_rule_files, run
from anchorai.reasoning.facts import build_facts
from anchorai.scenario.tables import read_tables

tables = read_tables()
scores = load_scores().set_index("member_id")
facts = build_facts(scores, load_features(), tables["interventions"], load_kg_facts())
rules = load_rule_files()


def probability(p):
    """Same rule as the web app (src/format.ts): a model probability never reads as 0% or 100%."""
    shown = round(p * 100)
    return ">99%" if shown >= 100 else "<1%" if shown <= 0 else f"{shown}%"


def member_facts_line(member_id, f):
    contract = "month-to-month" if f["contract_period"] == 1 else f"{f['contract_period']}-month"
    group = "attends group classes" if f.get("group_visits") == 1 else "no group classes"
    return (
        f"{member_id} profile: {contract} contract, {f['lifetime']} months as a member, "
        f"{f['avg_class_frequency_current_month']:.1f} visits/week this month vs {f['avg_class_frequency_total']:.1f} usual, {group}"
    )


def member_evidence(member_id):
    row = scores.loc[member_id]
    outcome = run(rules, facts[member_id], member_id)
    lines = [
        f"{member_id}: churn probability {probability(row['churn_prob'])} ({row['risk_band']} risk), segment '{row['segment']}'",
        member_facts_line(member_id, facts[member_id]),
    ]
    for name, weight in json.loads(row["top_drivers"])[:3]:
        lines.append(f"{member_id} risk driver: {LABELS.get(name, name)} ({'raises' if weight > 0 else 'lowers'} risk)")
    lines += [f"{member_id} rule {f.rule_id}: {f.description} -> {f.effect}" for f in outcome.firings]
    lines += [f"{member_id} allowed offer: {p.offer_id} via {p.channel} (rule {p.rule_id})" for p in outcome.allowed]
    lines += [f"{member_id} graph signal: {g}" for g in graph_signals(outcome.facts)]
    return lines


# A high-risk member who has an allowed action and a graph signal makes a good first test.
high = scores.index[scores["risk_band"] == "high"]
example = next(m for m in high if graph_signals(facts[m]) and run(rules, facts[m], m).allowed)
evidence = member_evidence(example)
print(example)
print("\n".join(f"[E{i}] {line}" for i, line in enumerate(evidence, 1)))

M0155
[E1] M0155: churn probability >99% (high risk), segment 'New and slipping'
[E2] M0155 profile: month-to-month contract, 0 months as a member, 0.6 visits/week this month vs 1.0 usual, attends group classes
[E3] M0155 risk driver: Drop in weekly visits (raises risk)
[E4] M0155 risk driver: Months as a member (raises risk)
[E5] M0155 risk driver: Age (raises risk)
[E6] M0155 rule D05_high_value_at_risk: High churn risk on a higher monthly fee (month-to-month or 6-month contract) -> assert high_value_at_risk=True
[E7] M0155 rule D02_new_member: In their first two months, where churn is 50-82% and falls below base rate by month 3 (c_rules) -> assert new_member=True
[E8] M0155 rule D06_social_member: Joined through a partner company or a friend, so social offers fit -> assert social_member=True
[E9] M0155 rule D07_lost_instructor: Most of their class visits were with an instructor who has left -> assert routine_disrupted=True
[E10] M0155 rule P03_disrupted_checkin: Members whose class 

## 3. Grounded prompt (draft)

Edit the wording here and re-run the tests below until all three behave.

In [5]:
DONT_KNOW = "I don't know based on the data I have."

SYSTEM = f"""You are AnchorAI's assistant for gym staff.
Answer ONLY from the evidence lines you are given. Each line starts with an id such as [E1].
After each sentence, cite the ids it is based on, e.g. [E1][E3].
If the evidence does not answer the question, reply exactly: {DONT_KNOW}
Never invent numbers, names, offers or reasons that are not in the evidence.
Use plain English, at most 80 words, no tables or headings."""


def ask(question, evidence_lines):
    block = "\n".join(f"[E{i}] {line}" for i, line in enumerate(evidence_lines, 1)) or "(no evidence)"
    start = time.perf_counter()
    answer = chat([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Question: {question}\n\nEvidence:\n{block}"},
    ])
    return answer.strip(), time.perf_counter() - start


def check(answer, evidence_lines):
    """Cheap faithfulness checks (the start of E8): citations valid, no numbers absent from evidence."""
    cited = {int(i) for i in re.findall(r"\[E(\d+)\]", answer)}
    numbers = set(re.findall(r"\d+(?:\.\d+)?%?", re.sub(r"\[E\d+\]", "", answer)))
    evidence_numbers = set(re.findall(r"\d+(?:\.\d+)?%?", " ".join(evidence_lines)))
    return {
        "says_dont_know": DONT_KNOW.lower() in answer.lower(),
        "citations": sorted(cited),
        "bad_citations": sorted(i for i in cited if not 1 <= i <= len(evidence_lines)),
        "numbers_not_in_evidence": sorted(numbers - evidence_numbers),
    }

### Test 1: a question the evidence answers

In [6]:
answer, secs = ask(f"Why is {example} at risk, and what should we offer them?", evidence)
print(answer, f"\n\n({secs:.1f}s)")
check(answer, evidence)

M0155 is at high risk, with a churn probability above 99% [E1]. Their weekly visits fell to 0.6 from a usual 1.0 [E2][E3], and they are a new month-to-month member [E2][E4]. Their age also raises risk [E5].

Their class routine was disrupted because all their class visits were with an instructor who has left [E9][E19].

Suggested offers:
- A personal check-in by SMS about alternative classes [E10][E15]
- A free trainer onboarding session by call [E11][E16]
- A bring-a-friend class pass by SMS, since they joined through a partner or friend [E8][E13][E17]
- A discounted month by call [E14][E18] 

(6.5s)


{'says_dont_know': False,
 'citations': [1, 2, 3, 4, 5, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19],
 'bad_citations': [],
 'numbers_not_in_evidence': []}

### Test 2: no evidence at all (should say it doesn't know)

In [7]:
answer, secs = ask(f"Why is {example} at risk?", [])
print(answer, f"({secs:.1f}s)")
check(answer, [])

I don't know based on the data I have. (2.8s)


{'says_dont_know': True,
 'citations': [],
 'bad_citations': [],
 'numbers_not_in_evidence': []}

### Test 3: evidence that doesn't cover the question (should refuse, not guess)

In [8]:
answer, secs = ask(f"What is {example}'s phone number and how much do they spend on supplements?", evidence)
print(answer, f"({secs:.1f}s)")
check(answer, evidence)

I don't know based on the data I have. (1.4s)


{'says_dont_know': True,
 'citations': [],
 'bad_citations': [],
 'numbers_not_in_evidence': []}

## 4. Decide

- Model: which one, and its latency
- Prompt wording: anything to change after the three tests?
- Temperature: keep 0.1?

Then this prompt moves into `anchorai/assistant/prompt.py`, and the notebook imports it from there.

## 5. Follow-up questions (multi-turn)

The chat sends earlier turns back with each question. A question with no intent of its own continues the previous
turn's intent and members, and earlier answers go to the LLM without their old evidence ids. This uses the package
code (`anchorai.assistant`), so it checks the prompt the app actually sends. Look for: the follow-up answer
reuses the same evidence, cites it, and doesn't add facts.

In [9]:
from anchorai.assistant.service import HistoryTurn, answer
from anchorai.db.session import get_engine
from sqlmodel import Session

with Session(get_engine()) as session:
    first = answer(session, f"Why is {example} at risk?", role="manager")
    turn = HistoryTurn(question=f"Why is {example} at risk?", answer=first.answer, intent=first.intent, slots=first.slots)
    second = answer(session, "Can you explain that more simply?", role="manager", history=[turn])
for r in (first, second):
    print(r.intent, "| follow-up:", r.follow_up, "|", r.slots.member_ids)
    print(r.answer, "\n")
check(second.answer, [e.text for e in second.evidence])

/home/aziz/Code/IRS-PM-2026-08-17-ISFT-GRP-16-AnchorAI/SystemCode/backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 33685.25it/s]


member_risk | follow-up: False | ['M0155']
M0155 is rated high risk, with a churn probability above 99%, and is in the 'New and slipping' segment [E1]. Key factors:

- Weekly visits have dropped to 0.6 this month from a usual 1.0 [E2][E3].
- They have 0 months as a member and are on a month-to-month contract [E2][E4].
- Their age raises their risk [E5].
- All of their class visits were with an instructor who has left [E6]. 

member_risk | follow-up: True | ['M0155']
M0155 is very likely to leave, with a churn probability above 99%. [E1]

- They are visiting less: 0.6 visits a week this month, down from their usual 1.0. [E2][E3]
- They are brand new, with 0 months as a member, and they're on a month-to-month contract. [E2][E4]
- Their age also adds to the risk. [E5]
- All of their class visits were with an instructor who has since left. [E6] 



{'says_dont_know': False,
 'citations': [1, 2, 3, 4, 5, 6],
 'bad_citations': [],
 'numbers_not_in_evidence': []}